In [1]:
import os
import json
import duckdb

# Setup paths
FHIR_DIR = "../synthea/output/fhir"
DB_PATH = "../data/omop_clinical.duckdb"

def extract_conditions(patient_file):
    """
    Reads a FHIR bundle and extracts condition (diagnosis) records.
    Returns a list of tuples ready for bulk database insertion.
    """
    file_path = os.path.join(FHIR_DIR, patient_file)
    with open(file_path, 'r', encoding='utf-8') as f:
        fhir_data = json.load(f)
        
    conditions = []
    
    for entry in fhir_data.get('entry', []):
        resource = entry.get('resource', {})
        
        # We only want Condition resources
        if resource.get('resourceType') == 'Condition':
            
            # Extract Patient Reference (and hash it to match the PERSON table)
            subject_ref = resource.get('subject', {}).get('reference', '')
            patient_source_id = subject_ref.replace('urn:uuid:', '')
            person_id = abs(hash(patient_source_id)) % (10**9)
            
            # Extract SNOMED Code and Text Description
            coding = resource.get('code', {}).get('coding', [])
            snomed_code = "0"
            condition_text = "Unknown"
            
            if coding:
                # Synthea usually provides SNOMED codes as the primary system
                snomed_code = coding[0].get('code', '0')
                condition_text = coding[0].get('display', 'Unknown')
                
            # Extract Start Date
            start_date = resource.get('onsetDateTime', '1900-01-01')[:10] 
            
            conditions.append((
                person_id,
                snomed_code,
                condition_text,
                start_date
            ))
            
    return conditions

print("⚙️ STARTING ETL PIPELINE (FHIR -> OMOP CONDITION_OCCURRENCE)\n" + "-"*50)

# 1. Extraction Phase (Python)
print("🔍 Extracting diagnoses from FHIR JSON files...")
json_files = [f for f in os.listdir(FHIR_DIR) if f.endswith('.json')]
all_conditions = []

for file in json_files:
    all_conditions.extend(extract_conditions(file))

print(f"📊 Extracted {len(all_conditions)} raw clinical conditions.")

# 2. Load & Transform Phase (DuckDB)
print("🔌 Connecting to DuckDB for vocabulary mapping...")

try:
    with duckdb.connect(DB_PATH) as con:
        
        # Load raw data into a temporary staging table
        con.execute("DROP TABLE IF EXISTS stg_condition")
        con.execute("""
            CREATE TEMPORARY TABLE stg_condition (
                person_id BIGINT,
                snomed_code VARCHAR,
                condition_text VARCHAR,
                start_date DATE
            )
        """)
        
        con.executemany("""
            INSERT INTO stg_condition VALUES (?, ?, ?, ?)
        """, all_conditions)
        
        print("⏳ Creating standard OMOP CONDITION_OCCURRENCE table...")
        con.execute("""
            CREATE TABLE IF NOT EXISTS condition_occurrence (
                condition_occurrence_id BIGINT PRIMARY KEY,
                person_id BIGINT,
                condition_concept_id INTEGER,
                condition_start_date DATE,
                condition_source_value VARCHAR,
                condition_source_concept_id VARCHAR
            )
        """)
        
        print("🧠 Performing SQL JOIN with OMOP Concept table to translate SNOMED codes...")
        # This is where the magic happens: mapping the raw SNOMED code to the official OMOP concept_id
        con.execute("""
            INSERT INTO condition_occurrence 
            SELECT 
                -- Generate a unique ID for each record
                ROW_NUMBER() OVER () + COALESCE((SELECT MAX(condition_occurrence_id) FROM condition_occurrence), 0) AS condition_occurrence_id,
                stg.person_id,
                COALESCE(c.concept_id, 0) AS condition_concept_id, -- 0 means 'No matching concept'
                stg.start_date AS condition_start_date,
                stg.condition_text AS condition_source_value,
                stg.snomed_code AS condition_source_concept_id
            FROM stg_condition stg
            LEFT JOIN concept c 
                ON stg.snomed_code = c.concept_code 
                AND c.vocabulary_id = 'SNOMED'
                AND c.domain_id = 'Condition'
        """)
        
        # Verification
        mapped_count = con.execute("SELECT COUNT(*) FROM condition_occurrence WHERE condition_concept_id != 0").fetchone()[0]
        unmapped_count = con.execute("SELECT COUNT(*) FROM condition_occurrence WHERE condition_concept_id = 0").fetchone()[0]
        
        print(f"\n✅ ETL Complete!")
        print(f" - Successfully mapped to OMOP Standards: {mapped_count} conditions")
        print(f" - Failed to map (Unknown/Custom): {unmapped_count} conditions")
        
        print("\n🔎 Sample of standard mapped conditions:")
        sample = con.execute("""
            SELECT person_id, condition_concept_id, condition_source_value, condition_start_date 
            FROM condition_occurrence 
            WHERE condition_concept_id != 0
            LIMIT 5
        """).fetchall()
        
        for row in sample:
            print(f" - Person: {row[0]:<10} | Concept ID: {row[1]:<8} | Date: {row[3]} | Diagnosis: {row[2]}")

except Exception as e:
    print(f"❌ Database error: {e}")

⚙️ STARTING ETL PIPELINE (FHIR -> OMOP CONDITION_OCCURRENCE)
--------------------------------------------------
🔍 Extracting diagnoses from FHIR JSON files...
📊 Extracted 1758 raw clinical conditions.
🔌 Connecting to DuckDB for vocabulary mapping...
⏳ Creating standard OMOP CONDITION_OCCURRENCE table...
🧠 Performing SQL JOIN with OMOP Concept table to translate SNOMED codes...

✅ ETL Complete!
 - Successfully mapped to OMOP Standards: 680 conditions
 - Failed to map (Unknown/Custom): 1078 conditions

🔎 Sample of standard mapped conditions:
 - Person: 315831628  | Concept ID: 81151    | Date: 2026-02-25 | Diagnosis: Sprain of ankle (disorder)
 - Person: 996266453  | Concept ID: 40481087 | Date: 2009-10-08 | Diagnosis: Viral sinusitis (disorder)
 - Person: 857089016  | Concept ID: 40486433 | Date: 2006-12-09 | Diagnosis: Perennial allergic rhinitis (disorder)
 - Person: 315831628  | Concept ID: 81151    | Date: 2024-12-01 | Diagnosis: Sprain of ankle (disorder)
 - Person: 996266453  | Co